<font color=red>**Danger zone:**</font> you'll be fine-tuning a model to generate positive, negative or even toxic reviews. We'll be doing this for fun, but this is also the technique for [review bombing](https://en.wikipedia.org/wiki/Review_bomb), bot farms on social media and other less than dignified stuff. It is ultimately your decision how you apply this knowledge, but before you choose, ask yourself: is this why you chose to learn ML?


# LLMs Alignment with Reinforcement Learning from human feedback (RLHF).

_based on the [original notebook](https://github.com/antndlcrx/oxford-llms-workshop/blob/main/materials/seminars/day_3/8_LLMs%20alignment%20with%20RLHF.ipynb) by Ilya Boytsov for the Oxford LLMs workshop_



In this session, you're gonna fine-tune a language model with reinforcement learning to make it generate good (or bad) reviews.

To perform RL-based fine-tuning, we'll use a new (in this course) library called [Transformer Reinforcement Learning (TRL)](https://huggingface.co/docs/trl). TRL implements the main reinforcement learning components of RLHF: reward modeling and fine-tuning with PPO.

![img](https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/TRL-readme.png)

In [1]:
# %pip install trl==0.7.1 transformers==4.33.1 datasets==2.14.4 peft==0.5.0
# %pip install trl==0.7.1

# %pip install trl==0.7.1 transformers==4.33.1 datasets==2.18.4 peft==0.5.0

# %pip install trl===0.7.4

### Tutorial: align the model to generate positive movie reviews

To see how TRL works, we'll use it to align GPT2 on IMDB dataset to generate positive (or negative) movie reviews. In fact, __it's your choice whether you want positive or negative reviews.__

But before you choose, let's take a look at the baseline model: a GPT-2 fine-tuned on generating arbitrary movie reviews.

In [2]:
MODEL_NAME="lvwerra/gpt2-imdb"
CACHE_DIR = ".cache/"

In [3]:
import torch
import transformers

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
main_tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
main_model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device, cache_dir=CACHE_DIR)

/home/yaslamov/personal_projects/nlp_course/week08_rlhf/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yaslamov/personal_projects/nlp_course/week08_rlhf/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/yaslamov/personal_projects/nlp_course/week08_rlhf/.venv/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/yaslamov/personal_projects/nlp_course/week08_rlhf/.venv/

In [4]:
inputs = main_tokenizer("The movie", return_tensors='pt').to(device)
generated_ids = main_model.generate(**inputs, max_new_tokens=50, do_sample=True)
print("\nGenerated text:", main_tokenizer.decode(generated_ids.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Generated text: The movie is just very boring and it really doesn't follow very well. So it's definitely not recommended.<|endoftext|>


If you run this cell a couple of times, you'll see that the model generates both positive, negative and neutral reviews in some proportion. What we're gonna do next is teach the model to generate more positive (or negative) reviews.

Similarly to InstructGPT, we're gonna do that in 2 stages:
- **train a reward model** to assign higher values to positive (or negative) reviews
- fine-tune the language model to **maximize that reward using [proximal policy optimization](https://openai.com/research/openai-baselines-ppo)**



## Load pre-trained reward model

In [5]:
MODEL_DIR = "reward_model/distilbert-base-cased/"

reward_model = transformers.AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
reward_model.to(device)

reward_tokenizer = transformers.AutoTokenizer.from_pretrained("distilbert-base-cased", cache_dir=".cache/")

Evaluate reward model

In [6]:
from typing import List
def compute_reward(texts: List[str]) -> torch.Tensor:
  inputs = reward_tokenizer(texts, truncation=True, padding=True, return_tensors='pt').to(device)
  with torch.no_grad():
    return reward_model(**inputs).logits[:, 0]

In [7]:
compute_reward("This movie sucked. It really was a waste of my life. The acting was atrocious, the plot completely implausible. Long, long story short, these people get \"terrorized\" by this pathetic \"crazed killer\", but completely fail to fight back in any manner. And this is after they take a raft on a camping trip, with no gear, and show up at a campsite that is already assembled and completely stocked with food and clothes and the daughters headphones. Additionally, after their boat goes missing, they panic that they're stuck in the woods, but then the daughters boyfriend just shows up and they apparently never consider that they could just hike out of the woods like he did to get to them. Like I said, this movie sucks. A complete joke. Don't let your girlfriend talk you into watching it.")


tensor([-5.2198], device='cuda:0')

In [8]:
compute_reward("Good: Engaging cinematic firefights, great presentation, vehicles are actually fun to drive, fairly appealing multiplayer, faithful to the movie, and the list goes on.<br /><br />Bad: Main missions are a bit short.<br /><br />This game defines what a \"good\" third person shooter(not necessarily a spy-game) is. Great firefights carry on the story and make you want to complete EVERY single mission through, and unlock all the genuine bonuses the game has to offer. The hype this game had, was lived up to, and I personally think you should buy it, and hook up with a couple of friends and play this one. Loads of fun. <br /><br />The sound in this game, is a rip-roaring achievement from a few previous bond games, and firing a weapon, really feels like you're firing a weapon. It ties in with the aspect that you are a deadly and ruthless spy.<br /><br />All in all, this game makes you excited and satisfied after you make it through, and some multiplayer that can compete with the standards of the crafty James Bond \"Nightfire\" game for gamecube.")

tensor([5.3564], device='cuda:0')

### Reward-guided generation (1 point)

If you did everything right, by now you should have a decent reward model. Before we use it for reinforcement learning, let's see if we can align model samples without any training.

To do so, you can use reward-guided inference: __generate N=16 samples, then select the one with the highest reward__ (according to your reward model).

For this problem, it's on you to demonstrate whether or not your code works. Find at least 5 neutral prompts such as "This movie is" (...), generate samples, rank them based on reward and show which samples get the highest reward.

Note: it is faster to generate samples in parallel, rather than sequentially, as follows:




In [12]:
def generate_best_review(start_str: str) -> str:
    best_reward = -1000
    best_sample = ''
    
    inputs = main_tokenizer([start_str] * 16, return_tensors='pt').to(device)
    for candidate in main_model.generate(**inputs, max_new_tokens=50, do_sample=True):
        sample = main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist())
        reward_input = reward_tokenizer(sample, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            reward = reward_model(**reward_input).logits[0, 0].item()
            if reward > best_reward:
                best_sample = sample
                best_reward = reward
                
    return best_sample
    

In [13]:
print(generate_best_review("It was"))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


It was very interesting. It was the first of the Gertrude van Rhoecker series, made in 1969, and this is really by far the best of them. The special effects are spectacular, and the story is gripping and interesting.<|endoftext|><|endoftext|>


In [14]:
print(generate_best_review("The movie"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The movie that it was in was the movie "Saving Private Ryan."<br /><br />But this is one of the better films of the decade. It offers some good bits of humor and a lot of great scenes from the movie. I liked


# Stage 2: fine-tune the main model with RL


For this tutorial, we will optimize GPT2 to produce positive IMDB movie reviews using the reward model you trained above.

Unlike supervised fine-tuning, RL allows model to generate it's own sentences on each training step. Then, it calculates the reward of those specific sentences, and finally, updates the model to increase the probability of sentences with high reward.

Thus, each RLHF consists of three stages: __Rollout__, __Evaluation__ and __Update__

<div style="text-align: center">
<img src='https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/gpt2_bert_training.png' width='600'>

The update stage depends on the specific RL algorithm. We'll be using Proximal Policy Optimization, or [PPO](https://arxiv.org/abs/1707.06347), similarly to what was used for InstructGPT.

Before we run those 3 stages, however, we need to create a dataset of "queries" - partial reviews in our case.

Create a dataset for RLHF

In [15]:
# Note: this code is specific to IMDB; you will need to re-write it for other tasks

from datasets import load_dataset
imdb = load_dataset("imdb", split="train", cache_dir=CACHE_DIR)

In [16]:
imdb[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [17]:
from trl.core import LengthSampler

dataset = imdb.filter(lambda row: len(row['text']) > 200, batched=False)
dataset = dataset.remove_columns(['label'])

sample_length = LengthSampler(2, 8)  # use the first 2-8 tokens as query

def select_query_and_tokenize(sample):
    query_ids = main_tokenizer.encode(sample["text"])[: sample_length()]
    sample["query"] = main_tokenizer.decode(query_ids)  # query is the only required column
    sample["input_ids"] = query_ids  # to avoid re-tokenizing later
    return sample  # we do not need the rest - it will be generated by the model

dataset = dataset.map(select_query_and_tokenize, batched=False)
dataset.set_format(type="torch")

In [18]:
len(dataset)

24895

In [20]:
dataset[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

Finally, we move to RL training. In this tutorial, we'll train LoRA adapters and not the full model.

In [21]:
import peft

model = transformers.GPT2LMHeadModel.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model_generation_config = model.generation_config

peft_config = peft.LoraConfig(
    task_type=peft.TaskType.CAUSAL_LM,
    inference_mode=False,
    r=32,
    lora_alpha=32,
    lora_dropout=0.1,
)

peft_model = peft.get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

/home/yaslamov/personal_projects/nlp_course/week08_rlhf/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/yaslamov/personal_projects/nlp_course/week08_rlhf/.venv/lib/python3.10/site-packages/peft/tuners/lora.py:475: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 1,179,648 || all params: 125,619,456 || trainable%: 0.939064725769868


In [22]:
import trl

model = trl.AutoModelForCausalLMWithValueHead.from_pretrained(peft_model, is_trainable=True, cache_dir=CACHE_DIR)
# model.generation_config = model_generation_config

ref_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
tokenizer.pad_token = tokenizer.eos_token

/home/yaslamov/personal_projects/nlp_course/week08_rlhf/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [23]:
# model.generation_config
model.config

GPT2Config {
  "_name_or_path": "lvwerra/gpt2-imdb",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "output_past": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "transformers_version": "4.33.1",
  "use_cache": true,
  "vocab_size": 50257
}

In [24]:
def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

In [25]:
config = trl.PPOConfig(
    # model_name = MODEL_NAME,
    # reward_model_path=reward_model.config._name_or_path,
    # model_adapter_name = peft_model.config._name_or_path,
    learning_rate=1.41e-5,
    gradient_accumulation_steps=1,
    batch_size=64,
    ppo_epochs=4,
    # ds3_gather_for_generation = False,
)

ppo_trainer = trl.PPOTrainer(
    config, model, ref_model, tokenizer, dataset=dataset, data_collator=collator
)


Same as before, trl has a special type of trainer that minimize PPO-specific pseudo-loss. You can read more on this trainer [here](https://huggingface.co/docs/trl/main/en/ppo_trainer).

In [ ]:
# training_args = trl.PPOConfig(
#     reward_model_path = reward_model.config._name_or_path,
#     # model_adapter_name=main_model.config._name_or_path,
#     gradient_accumulation_steps=1,
#     learning_rate=1.41e-5,
#     batch_size=64,
#     num_ppo_epochs=4,                 # PPO performs this many updates per training batch
# )

# ppo_trainer = trl.PPOTrainer(training_args, 
#                              model=main_model,
#                              ref_model=re,
#                              reward_model=reward_model,
#                              processing_class=main_tokenizer,
#                              train_dataset=imdb_for_rlhf, 
#                              peft_config=peft_config,
#                              data_collator=lambda data: dict((key, [d[key] for d in data]) for key in data[0])
# ) 
# # note: we pass main_model.model because PPOTrainer checks for one of several supported model types ...
# # ... main_model.model is a model with adapters, which is supported. main_model itself is a wrapper that is not supported

In [ ]:
# training_args = trl.PPOConfig(
#     model_name=main_model.config._name_or_path,
#     gradient_accumulation_steps=1,
#     learning_rate=1.41e-5,
#     batch_size=64,
#     ppo_epochs=4,                 # PPO performs this many updates per training batch
# )

# ppo_trainer = trl.PPOTrainer(
#     training_args, model=main_model.model, tokenizer=main_tokenizer,
#     dataset=imdb_for_rlhf, data_collator=lambda data: dict((key, [d[key] for d in data]) for key in data[0])
# )  # note: we pass main_model.model because PPOTrainer checks for one of several supported model types ...
# # ... main_model.model is a model with adapters, which is supported. main_model itself is a wrapper that is not supported

In [ ]:
from tqdm.auto import tqdm
max_steps = 50   # can be insufficient for some tasks - watch your learning curves
generation_kwargs = dict(
    min_length=-1, max_new_tokens=128, do_sample=True, top_k=0, top_p=1.0, pad_token_id=main_tokenizer.eos_token_id)
#                                  ^-- task-specific parameter!
with tqdm(enumerate(ppo_trainer.dataloader), total=max_steps) as progressbar:
  # note: ppo_trainer.dataloader is just a regular dataloader of queries, no RL-specific magic :)
  for epoch, batch in progressbar:
    if epoch >= max_steps:
        break

    # Rollout stage: generate continuations from batch queries using main_model
    response_tensors = ppo_trainer.generate(batch['input_ids'], **generation_kwargs)
    # ^-- list of tensors of token ids from main model tokenizer

    # de-tokenize responses to strings (since reward model uses a different tokenizer)
    batch["response"] = [main_tokenizer.decode(response.squeeze()) for response in response_tensors]
    # note: response_tensors already contain query tokens, so we don't need to add queries manually.
    # This may not be true for other tasks: check this manually by viewing batch["response"] and batch["query"]


    # Evaluation stage
    rewards = compute_reward(batch['response'])

    # Update stage
    stats = ppo_trainer.step(batch['input_ids'], response_tensors, list(rewards.split(1)))
    stats['rewards/mean'] = rewards.mean().item()

    print("-" * 30, 'STEP', epoch, '-' * 30)
    print(f'rewards/mean:\t{stats["rewards/mean"]:.9f}\t<---- average reward over this batch (higher=better, noisy)')
    print(f'ppo/returns/mean:\t{stats["ppo/returns/mean"]:.9f}\t<---- model-estimated average discounted reward')
    print(f'objective/kl:\t{stats["objective/kl"]:.9f}\t<---- how far we are from the original model (regularizer)')
    print()

    ppo_trainer.log_stats(stats, batch, list(rewards.split(1)))

  2%|▏         | 1/50 [00:47<38:36, 47.28s/it]

------------------------------ STEP 0 ------------------------------
rewards/mean:	-0.476456642	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-1.012933254	<---- model-estimated average discounted reward
objective/kl:	0.000000000	<---- how far we are from the original model (regularizer)



  4%|▍         | 2/50 [01:36<38:34, 48.22s/it]

------------------------------ STEP 1 ------------------------------
rewards/mean:	0.310531557	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	-0.712938428	<---- model-estimated average discounted reward
objective/kl:	0.104604252	<---- how far we are from the original model (regularizer)



  6%|▌         | 3/50 [02:24<37:45, 48.19s/it]

...

------------------------------ STEP 36 ------------------------------
rewards/mean:	4.609096527	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	3.131476641	<---- model-estimated average discounted reward
objective/kl:	12.525380135	<---- how far we are from the original model (regularizer)



 76%|███████▌  | 38/50 [23:05<06:23, 31.99s/it]

------------------------------ STEP 37 ------------------------------
rewards/mean:	4.485628128	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	3.085927486	<---- model-estimated average discounted reward
objective/kl:	11.782800674	<---- how far we are from the original model (regularizer)



 78%|███████▊  | 39/50 [23:40<06:01, 32.87s/it]

------------------------------ STEP 38 ------------------------------
rewards/mean:	4.589707375	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	3.183376789	<---- model-estimated average discounted reward
objective/kl:	11.831171036	<---- how far we are from the original model (regularizer)



## Main assignment - <u>actually</u> train the model (8 points)


Your main task for this week is to use the RLHF pipeline to train a model for a reward of your choice. Here's what you can choose from:

__A. Toxicity fine-tuning:__ train the model to be less (or more!) toxic. For this task, you may use the data from [jigsaw toxic comments](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge) and [lmsys/toxic-chat](https://huggingface.co/datasets/lmsys/toxic-chat),  or any other source. Alternatively, you may use toxicity scores from [oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1).


__B. Actual human feedback:__ use one of the existing datasets with pairwise human feedback to align your langauge model. You may use [anthropic's hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf), [OpenAssistant dataset](https://huggingface.co/datasets/OpenAssistant/oasst1) or any other data you see fit. You may also turn the tables and train the model to [minimize](https://habrastorage.org/getpro/geektimes/post_images/ac7/2ad/827/ac72ad82767d4132164a4b6b76196c42.jpg) human preferences, as long as your model does not degrade to gibberish.

__C. Controlled generation:__ Instead of training a reward model from human feedback, you may define the reward function as the text length (longer or shorter) or number of times the model uses specific words (e.g. "sorry", "apologize"). If you choose specific words, make sure the model generates them at least sometimes.

__Alternatively,__ you may choose a different task. However, unless your task is very similar to one of the above, there is a chance that it will be **significantly** harder to solve, requiring orders of magnitude more compute and tuning. If you are in doubt, please ask the course staff. If they are AFK (again >.<), please prefer one of the recommended tasks.


#### General tips & tricks


Things to look out for:
- during PPO stage, the reward model should be in eval mode (dropout disabled)
- make sure max_length and max_new_tokens are enough for your chosen dataset - at least most of the time
- when in doubt, view the data manually or inspect how the model performs on a few samples


We highly recommend that you manually check the performance after each sub-stage:
1. when you assembled the pairwise dataset, inspect a couple of from of *your* dataset class and detokenize them. Make sure that you-the-human understand why one sample was accepted and the other - rejected. At least most of the time. This also lets you spot tokenization/truncation errors.
2. after you trained a reward model, measure how accurate this model is in isolation. If your reward model is poor, any subsequent RLHF will also fail.
3. once you've trained the main model with RL, ask it to generate examples and explore how well it does. If it produces an obviously bad output, check if the reward model assigns high reward to that output. If yes, reward model is the culprit; if no, it's a question of better/longer PPO training.

__It is also a good idea to periodically print samples during training.__

__When stuck, simplify the problem.__ If you've spent a several hours enchanting the reward model but it still won't budge, try switching to a simple subtask. For instance, if you're training on hh-rlhf, try limiting it the dataset to 10% of the shortest sequences - they are typically easier to learn.


## Assignment stages (and grading)

Regardless of the specific task you chose, your solution needs to contain several parts that will be graded separately.


#### Stage 1: reward model (4 points)

Construct a dataset for training the reward model on your problem. Then, train a reward model on that dataset and evaluate how well can your model predict preferences on a hold-out (test) subset of your data.

Please make sure that the part of your notebook where you evaluate reward model is clearly visible and reasonably easy to read. And for all that is holy, do not call it IMDB unless it actually **is** data of imdb movie reviews :)

__Not all tasks require a reward model for later PPO fine-tuning.__ For instance, there's no reason to train a reward model if your reward equals sentence length. Likewise, toxicity reward can be estimated with a pre-trained toxicity classifier. __If your task does not require training a reward model, please train an unrelated model on [hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) as though you were solving assignment version B.__ This is for grading purposes only, you won't use this model for stage 2.


#### Stage 2: RL fine-tuning (4 points)

Once the reward model is ready - or you can compute rewards without a model - it is time to maximize that reward with PPO. Optionally, you may replace PPO with another RL algorithm (or unlikelihood learning scheme), but only if you're feeling adventurous.


First, you need to choose a language model to be fine-tuned. You may choose any model, but make sure that your model **can** generate the data in your format. For instance, [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) is a general purpose LM and may (or may not) need prompt engineering to generate chat assistant responses. For that reason, it is best if you **do not use `"lvwerra/gpt2-imdb"` unless you're generating only movie reviews**.



There are two "difficulty modes" for this task:
For the **easy mode**, use [gpt2-large](https://huggingface.co/gpt2-large) or [opt-1.3b](https://huggingface.co/facebook/opt-1.3b) with minimal code changes.
If you want the **Hard mode:** use a larger (e.g. 7B) model in combination with `load_in_4bit` and LoRA, the same way we did last week.
Some reasonable model choices are [LLaMA-7B](https://huggingface.co/Enoch/llama-7b-hf), [Falcon-7b](https://huggingface.co/tiiuae/falcon-7b), [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) for general-purpose LM or [guanaco-7b](https://huggingface.co/timdettmers/guanaco-7b), [vicuna-7b](https://huggingface.co/lmsys/vicuna-7b-v1.5) for chat-based tasks, though there are many more (see [leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)). In the hard mode, you will need to modify the training arguments to enable 4-bit fine-tuning. Furthermore, your experiments will take somewhat longer to complete. On the plus side, your model will produce significantly better results.

__High reward is not enough!__ RL algorithms are famous for [cheating their reward functions](https://openai.com/research/faulty-reward-functions). To ensure that your model is actually doing what you want it to do, you will need some additional evaluation. To get the full grade, provide at least 20 side-by-side examples of your fine-tuned model vs original model predictions and a short summary.

Alternatively, you may provide 5 examples and some extrinsic evaluation metric over many examples. For instance, you may use a different pre-trained toxicity score for option A. When dealing with human preferences, you may choose to [enlist actual humans](https://toloka.ai/) or [ask GPT4/Claude](https://arxiv.org/pdf/2304.03277.pdf) to compare your model's predictions. For task C, when optimizing for simple rewards like sentence lengths, it is enough to compare histograms of rewards (e.g. average lengths).










